# <center>**Travaux exploratoires : résumé formaté d'un texte</center>**

# <center>**VI. Fonction d'extraction structurée : version complète</center>**

*Ce fichier est généré sur Jupyter. Pour le faire fonctinner, il faut se placer dans l'environemment coreferee-env.*

*Date de dernière mise à jour : 06/08/2025*

**Contexte :**

  - dans le notebook *04_modele_llama3-70b-8192.ipynb* on a codé une fonction *extraction_structuree(nom_fichier)* qui en entrée prend le nom d'un fichier texte, et en sortie crée et enregistre un fichier .csv et un fichier .json qui stockent des extractions structurées des événements présents dans le fichier, sous la forme

    - résumé (5-6 mots)

    - lieu

    - moment

    - individus

    Cette fonction est en fait une surcouche d'une fonction extraire_faits(), présente dans le même notebook, qui à partir d'un texte relatant un événement rend en sortie l'extraction structurée décrite ci-dessus.

  - dans le notebook *05_decoupage_texte.ipynb* prend en entrée le nom d'un fichier texte et le nom d'un répertoire, et rend en sortie des blocs de ce texte au format .json, dans le répertoire donné en entrée.


**Méthodologie :**

- découper le texte avec la *fonction decouper_texte()*
- appliquer la fonction *extraction_faits()* à chaque bloc de texte

**Améliorations possibles :**

**Résultats :**


**Conclusion :**

**Import des librairies utiles**

# **I. Sans batch**

In [1]:
import pandas as pd
from openai import OpenAI
import time
import json
import os

In [2]:
!python -m spacy download fr_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 112.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


**Configuration du client OpenAI pour l'API Groq**

In [3]:
# 🔑 clé Groq
GROQ_API_KEY = "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"

# 📍 Configuration du client OpenAI pour l'API Groq
client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
    default_headers={"User-Agent": "pipeline-faits/1.0"}
)

**Fonction d'extraction des faits avec le modèle llama3-70b-8192**

In [ ]:
# 🧠 Fonction d'extraction à partir d’un texte
def extraire_faits(texte):
    prompt = f"""
Voici un texte :
{texte}

Tu dois extraire chaque fait important du texte sous forme d’une structure JSON avec ces champs :
- résumé : résumé du fait en 5 ou 6 mots
- lieu : lieu s'il y en a un, sinon "NA"
- moment : moment s'il y en a un, sinon "NA"
- individus : liste des individus impliqués s’il y en a, sinon "NA"

Retourne uniquement la liste JSON, sans explication.
"""
    try:
        response = client.chat.completions.create(
            model="llama3-70b-8192",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Erreur : {e}")
        return "ERREUR"

**Fonction de découpage de texte avec SpaCy**

Anciene version : conduit à beaucoup de blocs pour un fichier un peu long

In [ ]:
def decouper_texte(nom_fichier):
    import spacy
    from sentence_transformers import SentenceTransformer, util

    # Charger les modèles
    nlp = spacy.load("fr_core_news_sm")
    model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

    # Lire le texte
    with open(nom_fichier, encoding="utf-8") as f:
        texte = f.read()

    # Segmenter en phrases
    doc = nlp(texte)
    phrases = [sent.text.strip() for sent in doc.sents if sent.text.strip()]

    # Embeddings
    embeddings = model.encode(phrases, convert_to_tensor=True)

    seuil_similarite = 0.65
    blocs = []
    bloc_courant = [phrases[0]]

    for i in range(1, len(phrases)):
        sim = util.pytorch_cos_sim(embeddings[i], embeddings[i - 1]).item()
        if sim >= seuil_similarite:
            bloc_courant.append(phrases[i])
        else:
            blocs.append(" ".join(bloc_courant))
            bloc_courant = [phrases[i]]
    if bloc_courant:
        blocs.append(" ".join(bloc_courant))

    return blocs


Nouvelle version : réduit le nombre de blocs et donc réduit le temps de traitement

In [ ]:
def decouper_texte(nom_fichier, target_chars=2000, overlap=200, min_chars=400):
    texte = open(nom_fichier, "r", encoding="utf-8").read()
    morceaux = []
    i = 0
    n = len(texte)
    while i < n:
        j = min(i + target_chars, n)
        chunk = texte[i:j]
        # évite d'éclater une phrase au milieu
        if j < n:
            fin = chunk.rfind(". ")
            if fin > min_chars:  # ne recule pas trop
                chunk = chunk[:fin+1]
                j = i + len(chunk)
        if len(chunk) >= min_chars:
            morceaux.append(chunk)
        i = max(j - overlap, j)  # petit recouvrement
    return morceaux

**Fonction d'extraction de faits pour des textes de longueur quelconque**

<u>Principe : </u>
- découper le texte avec la fonction *decouper_texte()*
- appliquer la *fonction extraction_faits()* à chaque bloc de texte

Ancienne version, pas optimisée

In [ ]:
def extraire_faits_depuis_texte_long(nom_fichier):
    import pandas as pd
    import json
    import time

    print("⏳ Découpage du texte en blocs sémantiques...")
    blocs = decouper_texte(nom_fichier)
    print(f"✅ {len(blocs)} blocs détectés.")

    lignes = []
    for i, bloc in enumerate(blocs):
        print(f"\n🔎 Extraction des faits du bloc {i+1}/{len(blocs)}")
        texte = bloc.strip()
        try:
            faits_json = extraire_faits(texte)
            faits = json.loads(faits_json)
            for fait in faits:
                lignes.append({
                    "bloc": i + 1,
                    "texte": texte,
                    "résumé": fait.get("résumé", "NA"),
                    "lieu": fait.get("lieu", "NA"),
                    "moment": fait.get("moment", "NA"),
                    "individus": fait.get("individus", "NA")
                })
        except Exception as e:
            print(f"⚠️ Erreur sur le bloc {i+1} : {e}")
            lignes.append({
                "bloc": i + 1,
                "texte": texte,
                "résumé": "ERREUR",
                "lieu": "NA",
                "moment": "NA",
                "individus": "NA"
            })
        time.sleep(1)  # Pour respecter les limites API

    # Export CSV
    df = pd.DataFrame(lignes)
    df.to_csv("faits_extraits.csv", index=False)
    print("💾 Fichier CSV enregistré : faits_extraits.csv")

    # Export JSON
    with open("faits_extraits.json", "w", encoding="utf-8") as f:
        json.dump(lignes, f, indent=2, ensure_ascii=False)
    print("💾 Fichier JSON enregistré : faits_extraits.json")

Nouvelle version : on parallélise le calcul, en limitant toutefois le nombre de requêtes simultanées

- max_workers=6 et qps=3 limite ~3 requêtes/sec globales, 6 en parallèle (adapter selon le quota API).

- on écrit un checkpoint .jsonl au fil de l’eau → si Colab coupe, on ne perd pas tout.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import json, pandas as pd, time, math

def extraire_faits_depuis_texte_long(nom_fichier, max_workers=6, qps=3, checkpoint="checkpoint.jsonl"):
    blocs = decouper_texte(nom_fichier)
    print(f"✅ {len(blocs)} blocs détectés.")

    lignes = []
    start_time = time.time()
    last_tick = [0.0]  # pour limiter le débit global

    def call_api(i, texte):
        # rate limit global simple
        nonlocal last_tick
        while True:
            now = time.time()
            if now - last_tick[0] >= 1.0 / qps:
                last_tick[0] = now
                break
            time.sleep(0.01)

        try:
            faits_json = extraire_faits(texte)  # TA fonction
            faits = json.loads(faits_json)
            out = []
            for fait in faits:
                out.append({
                    "bloc": i + 1,
                    "texte": texte,
                    "résumé": fait.get("résumé", "NA"),
                    "lieu": fait.get("lieu", "NA"),
                    "moment": fait.get("moment", "NA"),
                    "individus": fait.get("individus", "NA")
                })
            return out, None
        except Exception as e:
            return [{
                "bloc": i + 1,
                "texte": texte,
                "résumé": "ERREUR",
                "lieu": "NA",
                "moment": "NA",
                "individus": "NA"
            }], str(e)

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(call_api, i, bloc.strip()): i for i, bloc in enumerate(blocs)}
        for fut in as_completed(futures):
            i = futures[fut]
            res, err = fut.result()
            lignes.extend(res)
            # checkpoint incrémental
            with open(checkpoint, "a", encoding="utf-8") as f:
                for row in res:
                    f.write(json.dumps(row, ensure_ascii=False) + "\n")
            if err:
                print(f"⚠️ Erreur sur le bloc {i+1}: {err}")
            if (i+1) % 50 == 0:
                elapsed = time.time() - start_time
                done = i+1
                eta = elapsed/done * (len(blocs)-done)
                print(f"Progression {done}/{len(blocs)} | ~ETA {int(eta/60)} min")

    # Export finaux (à partir du checkpoint pour robustesse)
    df = pd.DataFrame(lignes)
    df.to_csv("faits_extraits.csv", index=False)
    with open("faits_extraits.json", "w", encoding="utf-8") as f:
        json.dump(lignes, f, indent=2, ensure_ascii=False)
    print("💾 faits_extraits.csv / faits_extraits.json écrits.")


**On upolad un fichier long de textes**

In [ ]:
from google.colab import files

uploaded = files.upload() # on upolad le fichier fichier_textes_naturels.csv

Saving recueil_nouvelles.txt to recueil_nouvelles.txt


**On lance la fonction *extraire_faits_depuis_texte_long()* sur ce fichier**

In [ ]:
import time

start_time = time.time()

extraire_faits_depuis_texte_long("recueil_nouvelles.txt")

end_time = time.time()
print(f"⏱️ Temps d'exécution : {end_time - start_time:.2f} secondes")

⏳ Découpage du texte en blocs sémantiques...


FileNotFoundError: [Errno 2] No such file or directory: 'recueil_nouvelles.txt'

In [ ]:
from google.colab import files

# Téléchargement du CSV
files.download("faits_extraits.csv")

# Téléchargement du JSON
files.download("faits_extraits.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **II. Version batchée**

On utilise le client OpenAI pointé vers Groq (base_url="https://api.groq.com/openai/v1"), avec la clé dans GROQ_API_KEY.

**Objectif :** réduire le nombre d’appels réseau en envoyant plusieurs blocs de texte en une seule requête (batch).

**Pipeline complète :**

<center>découpe du fichier → batching → appel modèle → checkpoint → export CSV/JSON</center>

In [4]:
import os, json, time, math
from openai import OpenAI

**Initialisation du client**

In [5]:
# 🔑 clé Groq
GROQ_API_KEY = "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"

# 📍 Configuration du client OpenAI pour l'API Groq
client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
    default_headers={"User-Agent": "pipeline-faits/1.0"})

**Fonction *safe_json_loads()* :**

- charger du JSON potentiellement « sale » renvoyé par le modèle (espaces, BOM, texte parasite).

- faire une tentative de rattrapage si le modèle a ajouté un peu de texte autour et que le vrai JSON commence par [ … ].

**Entrées :**

- s (str) : texte brut retourné par le modèle.

**Sorties :**

- objet Python (liste/dict) parsé depuis s.

**Cas limites et remarques :**

- si le modèle envoie autre chose qu’un JSON, on lève une erreur claire.

- le fallback [ ... ] ne gère que des tableaux JSON (c’est notre format attendu). Si la sortie est un objet {}, on n’essaie pas de le rattraper.

- pour plus robuste encore, on pourrait ajouter un rattrapage {…}.

**Parsing d'une chaine JSON**


**But :** parser une chaîne JSON, avec un plan de secours si le modèle a mis du texte autour du vrai tableau JSON.

Étape 1 : on refuse None.

Étape 2 : on nettoie les bords (BOM/espaces/newlines/tabs).

Étape 3 : json.loads(s) → si OK, on retourne.

Fallback : on extrait le plus grand [ ... ] et on tente de le parser.

Échec total : on lève une ValueError lisible avec un extrait pour t’aider à comprendre ce qui cloche.

In [6]:
def safe_json_loads(s: str):
    """
    Chargement JSON robuste :
    - strip BOM/espaces
    - tente un parsing, sinon lève une ValueError lisible
    """
    if s is None:
        raise ValueError("Réponse vide du modèle.")
    s = s.strip("\ufeff \n\t")
    try:
        return json.loads(s)
    except Exception as e:
        # Petit secours: tenter d'extraire le bloc JSON principal s'il y a du texte parasite
        start = s.find("[")
        end = s.rfind("]")
        if start != -1 and end != -1 and end > start:
            try:
                return json.loads(s[start:end+1])
            except:
                pass
        raise ValueError(f"JSON invalide. Extrait début: {s[:200]!r} ... erreur: {e}")

**Prompts**

In [7]:
SYSTEM_BATCH = (
    "Tu es un extracteur de faits. "
    "Tu reçois une liste d'objets JSON {id, texte}. "
    "Pour chaque objet d'entrée, tu dois retourner un objet sortie avec: "
    "{id, faits}. 'faits' est une liste de dictionnaires, chacun avec EXACTEMENT les clés: "
    "résumé, lieu, moment, individus. "
    "Réponds STRICTEMENT en JSON, sans texte autour."
)


In [8]:
def build_user_message_for_batch(items):
    """
    items = liste de dicts: {"id": int, "texte": str}
    On envoie ces items au modèle, en demandant une structure bien définie en sortie.
    """
    payload = {
        "instructions": (
            "Analyse chaque 'texte' indépendamment. Ne mélange pas les faits entre items. "
            "Renvoie un tableau JSON d'objets, dans le même ordre que l'entrée, avec la forme:\n"
            "[\n"
            "  {\"id\": <id>, \"faits\": [\n"
            "    {\"résumé\": \"...\", \"lieu\": \"...\", \"moment\": \"...\", \"individus\": \"...\"},\n"
            "    ...\n"
            "  ]},\n"
            "  ...\n"
            "]\n"
            "Si tu ne trouves aucun fait pour un item, renvoie \"faits\": []."
        ),
        "items": items
    }
    return json.dumps(payload, ensure_ascii=False, indent=None)

**Appel batch unique**

In [9]:
def extraire_faits_batch(items, model="llama-3.3-70b-versatile", max_completion_tokens=2000, temperature=0.1):
    """
    Appelle le modèle UNE FOIS pour N textes (batch).
    - items: liste de dicts {"id": int, "texte": str} (5–10 items conseillé)
    - Retourne: dict[id] -> list[faits]
    """
    user_content = build_user_message_for_batch(items)
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role":"system","content": SYSTEM_BATCH},
            {"role":"user","content": user_content},
        ],
        temperature=temperature,
        max_completion_tokens=max_completion_tokens,
    )
    content = resp.choices[0].message.content
    parsed = safe_json_loads(content)

    # Normaliser en dict id -> list de faits
    out = {}
    if not isinstance(parsed, list):
        raise ValueError("La sortie du modèle n'est pas une liste JSON.")
    for obj in parsed:
        _id = obj.get("id")
        faits = obj.get("faits", [])
        if _id is None:
            # si le modèle a oublié l'id, on ignore cet objet
            continue
        # Valider structure minimale des faits
        cleaned = []
        for f in faits:
            cleaned.append({
                "résumé": f.get("résumé", "NA"),
                "lieu": f.get("lieu", "NA"),
                "moment": f.get("moment", "NA"),
                "individus": f.get("individus", "NA"),
            })
        out[_id] = cleaned
    return out

**Découpage du texte (chunking)**

In [10]:
def decouper_texte(nom_fichier, target_chars=2000, overlap=200, min_chars=400):
    """
    Chunking pragmatique: vise ~2000 caractères par bloc, overlap léger pour ne pas couper des faits.
    """
    with open(nom_fichier, "r", encoding="utf-8") as f:
        texte = f.read()
    morceaux, i, n = [], 0, len(texte)
    while i < n:
        j = min(i + target_chars, n)
        chunk = texte[i:j]
        if j < n:
            fin = chunk.rfind(". ")
            if fin > min_chars:
                chunk = chunk[:fin+1]
                j = i + len(chunk)
        if len(chunk) >= min_chars:
            morceaux.append(chunk)
        i = max(j - overlap, j)
    return morceaux

**Orchestrateur batch**

In [14]:
import pandas as pd

def extraire_faits_depuis_texte_long_batch(
    nom_fichier,
    batch_size=8,
    model="llama-3.3-70b-versatile",
    target_chars=2000,
    overlap=200,
    min_chars=400,
    checkpoint="checkpoint.jsonl",
    qps=0.8,             # limite ~0.8 requête/seconde (1 req toutes ~1.25 s)
    max_completion_tokens=2000,
    temperature=0.1,
    save_drive=False,    # si True, copie vers /content/drive/MyDrive/
):
    """
    Pipeline complet:
    - découpe le fichier en blocs
    - regroupe en batchs (8 par défaut)
    - UN appel modèle par batch (donc ~1/8 du nombre d'appels vs séquentiel)
    - écrit un checkpoint .jsonl au fil de l’eau (reprise possible)
    - export final CSV + JSON

    Retourne: DataFrame final.
    """
    blocs = decouper_texte(nom_fichier, target_chars, overlap, min_chars)
    print(f"✅ {len(blocs)} blocs détectés.")
    lignes = []
    last_call = 0.0

    # Efface l'ancien checkpoint s'il existe (optionnel)
    try:
        import os
        if os.path.exists(checkpoint):
            os.remove(checkpoint)
    except Exception:
        pass

    # Boucle par batchs
    for start in range(0, len(blocs), batch_size):
        end = min(start + batch_size, len(blocs))
        batch_items = [{"id": i, "texte": blocs[i].strip()} for i in range(start, end)]

        # Rate limit simple (QPS global)
        now = time.time()
        delta = 1.0 / qps - (now - last_call)
        if delta > 0:
            time.sleep(delta)
        last_call = time.time()

        # Appel batch
        try:
            res = extraire_faits_batch(
                batch_items,
                model=model,
                max_completion_tokens=max_completion_tokens,
                temperature=temperature,
            )
        except Exception as e:
            print(f"⚠️ Erreur batch {start+1}-{end}: {e}")
            # on met des lignes "ERREUR" pour ce batch, afin de ne pas bloquer toute la pipeline
            for it in batch_items:
                lignes.append({
                    "bloc": it["id"] + 1,
                    "texte": it["texte"],
                    "résumé": "ERREUR",
                    "lieu": "NA",
                    "moment": "NA",
                    "individus": "NA",
                })
            continue

        # Normalisation → lignes
        for it in batch_items:
            faits = res.get(it["id"], [])
            if not faits:
                # zéro fait → on écrit tout de même une ligne vide (optionnel)
                lignes.append({
                    "bloc": it["id"] + 1,
                    "texte": it["texte"],
                    "résumé": "",
                    "lieu": "",
                    "moment": "",
                    "individus": "",
                })
            else:
                for f in faits:
                    lignes.append({
                        "bloc": it["id"] + 1,
                        "texte": it["texte"],
                        "résumé": f.get("résumé", "NA"),
                        "lieu": f.get("lieu", "NA"),
                        "moment": f.get("moment", "NA"),
                        "individus": f.get("individus", "NA"),
                    })

        # Checkpoint incrémental
        with open(checkpoint, "a", encoding="utf-8") as f:
            for row in lignes[-max(1, len(lignes)) :]:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")

        # Petit ETA
        done = end
        pct = 100.0 * done / len(blocs)
        print(f"Batch {start+1}-{end} traité | {done}/{len(blocs)} blocs ({pct:.1f}%)")

    # Exports finaux
    df = pd.DataFrame(lignes)
    df.to_csv("faits_extraits.csv", index=False)
    with open("faits_extraits.json", "w", encoding="utf-8") as f:
        json.dump(lignes, f, indent=2, ensure_ascii=False)
    print("💾 faits_extraits.csv / faits_extraits.json écrits.")

    # Option: copie vers Google Drive
    if save_drive:
        try:
            from google.colab import drive
            drive.mount('/content/drive', force_remount=False)
            os.system("cp faits_extraits.csv /content/drive/MyDrive/")
            os.system("cp faits_extraits.json /content/drive/MyDrive/")
            print("📁 Copié vers Google Drive/MyDrive/")
        except Exception as e:
            print(f"⚠️ Copie vers Drive échouée: {e}")

    # Aperçu
    print("\n🔹 Aperçu (3 lignes):")
    for _, row in df.head(3).iterrows():
        print(dict(row))

    return df

**Application à un texte long**

In [12]:
from google.colab import files

uploaded = files.upload() # on upolad le fichier recueil_nouvelles.csv

Saving recueil_nouvelles.txt to recueil_nouvelles.txt


In [15]:
extraire_faits_depuis_texte_long_batch(nom_fichier="recueil_nouvelles.txt")

✅ 57 blocs détectés.
Batch 1-8 traité | 8/57 blocs (14.0%)
Batch 9-16 traité | 16/57 blocs (28.1%)
Batch 17-24 traité | 24/57 blocs (42.1%)
Batch 25-32 traité | 32/57 blocs (56.1%)
Batch 33-40 traité | 40/57 blocs (70.2%)
Batch 41-48 traité | 48/57 blocs (84.2%)
Batch 49-56 traité | 56/57 blocs (98.2%)
Batch 57-57 traité | 57/57 blocs (100.0%)
💾 faits_extraits.csv / faits_extraits.json écrits.

🔹 Aperçu (3 lignes):
{'bloc': 1, 'texte': "Fatale erreur\nFredric Brown\nM. Walter Baxter était un grand lecteur de romans policiers depuis de longues années. Le jour où il\ndécida d'assassiner son oncle, il savait donc qu'il ne devrait pas commettre le moindre impair.\nIl savait aussi que pour éviter toute possibilité d'erreur, le mot d'ordre devait être « simplicité ». Une\nrigoureuse simplicité. Pas d'alibi préparé à l'avance et qui risque toujours de ne pas tenir. Pas de modus\noperandi compliqué. Pas de fausses pistes manigancées.\nSi, quand même, une fausse piste, mais petite. Toute simple

,bloc,texte,résumé,lieu,moment,individus
0,1,Fatale erreur\nFredric Brown\nM. Walter Baxter...,M. Walter Baxter décide d'assassiner son oncle,La maison de l'oncle,Une nuit,"M. Walter Baxter, son oncle"
1,1,Fatale erreur\nFredric Brown\nM. Walter Baxter...,M. Walter Baxter cambriole la maison de son oncle,La maison de l'oncle,La même nuit,"M. Walter Baxter, son oncle"
2,1,Fatale erreur\nFredric Brown\nM. Walter Baxter...,M. Walter Baxter est arrêté par la police,La maison de M. Walter Baxter,Le lendemain,"M. Walter Baxter, le shérif, son adjoint"
3,2,Il s'était débarrassé de l'argent et de la pin...,M. Walter Baxter se débarrasse de l'argent et ...,La maison de M. Walter Baxter,Après le meurtre,M. Walter Baxter
4,2,Il s'était débarrassé de l'argent et de la pin...,M. Walter Baxter est arrêté par la police,La maison de M. Walter Baxter,Peu de temps après,"M. Walter Baxter, le shérif, son adjoint"
...,...,...,...,...,...,...
106,55,"Le\ndragon chargea, désarçonna le cavalier, le...",Le dragon attaque les chevaliers,La lande,Une nuit,Les deux chevaliers et le dragon
107,55,"Le\ndragon chargea, désarçonna le cavalier, le...",Les chevaliers sont vaincus par le dragon,La lande,Une nuit,Les deux chevaliers et le dragon
108,56,Au moindre\ndoute le vieux se mettait à hurler...,Un vieil homme aveugle sélectionne des personn...,Son château,Un jour,Le vieil homme et les candidats
109,56,Au moindre\ndoute le vieux se mettait à hurler...,Le vieil homme subit une opération pour recouv...,Son château,Un jour,Le vieil homme et le chirurgien


In [ ]:
import time

start_time = time.time()

extraire_faits_depuis_texte_long_batch(nom_fichier="recueil_nouvelles.txt")

end_time = time.time()
print(f"⏱️ Temps d'exécution : {end_time - start_time:.2f} secondes")

In [16]:
import os, json, pandas as pd

# Vérifier la présence et la taille
for p in ["faits_extraits.csv", "faits_extraits.json"]:
    print(p, "→", "OK" if os.path.exists(p) else "ABSENT",
          "| taille:", os.path.getsize(p) if os.path.exists(p) else 0, "octets")

# Lire le CSV
df = pd.read_csv("faits_extraits.csv")
display(df.head(10))           # aperçu

# Lire le JSON (liste de dicts)
with open("faits_extraits.json", "r", encoding="utf-8") as f:
    data = json.load(f)
print("JSON: nb lignes =", len(data))
data[:2]                        # aperçu 2 premières lignes

faits_extraits.csv → OK | taille: 231937 octets
faits_extraits.json → OK | taille: 246066 octets


,bloc,texte,résumé,lieu,moment,individus
0,1,Fatale erreur\nFredric Brown\nM. Walter Baxter...,M. Walter Baxter décide d'assassiner son oncle,La maison de l'oncle,Une nuit,"M. Walter Baxter, son oncle"
1,1,Fatale erreur\nFredric Brown\nM. Walter Baxter...,M. Walter Baxter cambriole la maison de son oncle,La maison de l'oncle,La même nuit,"M. Walter Baxter, son oncle"
2,1,Fatale erreur\nFredric Brown\nM. Walter Baxter...,M. Walter Baxter est arrêté par la police,La maison de M. Walter Baxter,Le lendemain,"M. Walter Baxter, le shérif, son adjoint"
3,2,Il s'était débarrassé de l'argent et de la pin...,M. Walter Baxter se débarrasse de l'argent et ...,La maison de M. Walter Baxter,Après le meurtre,M. Walter Baxter
4,2,Il s'était débarrassé de l'argent et de la pin...,M. Walter Baxter est arrêté par la police,La maison de M. Walter Baxter,Peu de temps après,"M. Walter Baxter, le shérif, son adjoint"
5,3,Je désigne la fenêtre du\npremier étage de la ...,Le narrateur rencontre Irène,Le parc Monceau,Un soir de printemps,"Le narrateur, Irène"
6,3,Je désigne la fenêtre du\npremier étage de la ...,Le narrateur et Irène se rencontrent à nouveau,Le parc Monceau,Le lendemain,"Le narrateur, Irène"
7,4,Je suis d’une nature\nassez sensible : je supp...,Le narrateur achète des fleurs pour Irène,Un fleuriste près du parc Monceau,Un soir de printemps,Le narrateur
8,4,Je suis d’une nature\nassez sensible : je supp...,Le narrateur donne les fleurs à Irène,Le parc Monceau,Le même soir,"Le narrateur, Irène"
9,5,Je lui souris et m’éloignai sans me retourner:...,Le narrateur et Irène se rencontrent à nouveau,Le parc Monceau,Plusieurs jours après,"Le narrateur, Irène"


JSON: nb lignes = 111


[{'bloc': 1,
  'texte': "Fatale erreur\nFredric Brown\nM. Walter Baxter était un grand lecteur de romans policiers depuis de longues années. Le jour où il\ndécida d'assassiner son oncle, il savait donc qu'il ne devrait pas commettre le moindre impair.\nIl savait aussi que pour éviter toute possibilité d'erreur, le mot d'ordre devait être « simplicité ». Une\nrigoureuse simplicité. Pas d'alibi préparé à l'avance et qui risque toujours de ne pas tenir. Pas de modus\noperandi compliqué. Pas de fausses pistes manigancées.\nSi, quand même, une fausse piste, mais petite. Toute simple. Il faudrait qu'il cambriole la maison de son\noncle, et qu'il emporte tout l'argent liquide qu'il y trouverait, de telle manière que le meurtre apparaisse\ncomme un cambriolage ayant mal tourné. Sans cela, unique héritier de son oncle, il se désignerait trop\ncomme suspect numéro un.\nIl prit tout son temps pour faire l'emplette d'une pince-monseigneur dans des conditions rendant\nimpossible l'identification de

In [ ]:
from google.colab import files

# Télécharger un fichier
files.download('mon_fichier.txt')  # Remplace par ton nom de fichier
